# 11차수 자가학습 정답
문제를 푼 뒤 실행해 검산하세요.

## 작성 프롬프트
문제를 푼 뒤 검산할 수 있도록 새 가상 자료를 처음부터 다시 생성하고 교차표·주변합·전체·행·열 백분율과 그림을 계산해 줘. 결측은 미분류로 따로 세고 비율은 유효 기록을 분모로 해 줘. 일반배송 지연 칸의 폭·높이·면적과 57.1%의 분모도 설명해 줘.

In [ ]:
import numpy as np
import pandas as pd

# 교육용 가상 표본. 한 행은 주문 한 건입니다.
groups = [("빠른배송", 18, 28), ("빠른배송", 30, 6),
          ("빠른배송", 38, 6), ("빠른배송", None, 1),
          ("일반배송", 25, 12), ("일반배송", 41, 8),
          ("일반배송", None, 1)]
orders = pd.DataFrame(
    [(method, minutes) for method, minutes, count in groups for _ in range(count)],
    columns=["method", "minutes"]
)
orders.insert(0, "order_id", range(1, len(orders) + 1))
print(orders.head())
print("전체 기록 수:", len(orders))


orders["late"] = np.select(
    [orders["minutes"].isna(), orders["minutes"].gt(30)],
    ["미분류", "지연"], default="정시"
)
missing = orders["late"].eq("미분류")
print("미분류 기록:", int(missing.sum()))
print(orders.loc[missing].groupby("method").size())
valid = orders.loc[~missing].copy()
methods = ["빠른배송", "일반배송"]
statuses = ["지연", "정시"]
tab = pd.crosstab(valid["method"], valid["late"]).reindex(
    index=methods, columns=statuses, fill_value=0
)
with_totals = tab.copy()
with_totals["합계"] = tab.sum(axis=1)
with_totals.loc["합계"] = with_totals.sum(axis=0)
total_pct = 100 * tab / tab.to_numpy().sum()
row_pct = 100 * tab.div(tab.sum(axis=1), axis=0)
col_pct = 100 * tab.div(tab.sum(axis=0), axis=1)
for title, values in [("빈도표와 주변합", with_totals), ("전체 백분율", total_pct),
                      ("행 백분율", row_pct), ("열 백분율", col_pct)]:
    print(title)
    print(values.round(2))

# 독립 검산. 표의 반올림 전 값으로 확인합니다.
assert orders["order_id"].nunique() == 62
assert missing.sum() == 2 and len(valid) == 60
assert tab.to_numpy().tolist() == [[6, 34], [8, 12]]
assert tab.to_numpy().sum() == len(valid)
assert np.allclose(row_pct.sum(axis=1), 100)
assert np.allclose(col_pct.sum(axis=0), 100)
assert np.isclose(total_pct.to_numpy().sum(), 100)

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Colab 기본 글꼴로 읽을 수 있도록 그림에는 영문 범주명을 사용합니다.
labels = ["Fast", "Regular"]
late_counts = tab["지연"].to_numpy()
late_rates = row_pct["지연"].to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
for ax, values, ylabel, suffix in [(axes[0], late_counts, "Late orders", ""),
                                   (axes[1], late_rates, "Late share within method (%)", "%")]:
    bars = ax.bar(labels, values, color=["#074c8c", "#dc6254"])
    ax.bar_label(bars, labels=[f"{v:g}{suffix}" for v in values], padding=3)
    ax.set_ylim(0, max(values) * 1.3)
    ax.set_ylabel(ylabel)

counts = tab.to_numpy()
widths = counts.sum(axis=1) / counts.sum()
heights = counts / counts.sum(axis=1, keepdims=True)
left = 0.0
for i, width in enumerate(widths):
    bottom = 0.0
    for j, height in enumerate(heights[i]):
        axes[2].add_patch(Rectangle((left, bottom), width, height,
                                   facecolor=["#dc6254", "#b6d2ea"][j], edgecolor="white"))
        axes[2].text(left + width / 2, bottom + height / 2,
                     f"{['Late', 'On time'][j]} {counts[i,j]}", ha="center", va="center")
        bottom += height
    left += width
axes[2].set(xlim=(0, 1), ylim=(0, 1), xticks=[1/3, 5/6], xticklabels=labels,
            ylabel="Within-method share", title="Mosaic")
plt.show()

width, height = widths[1], heights[1, 0]
print("일반배송 지연 칸의 폭·높이·면적:", width, height, width * height)
assert np.isclose(width * height, 8 / 60)
print("전체 지연 비율:", 100 * tab["지연"].sum() / len(valid))
print("일반배송 지연 비율은 8/20 = 40%입니다.")
print("57.1%는 지연 14건 중 일반배송 8건의 비중입니다.")
